In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import json
from nltk.tokenize import RegexpTokenizer

1. As per the JD only who are in India and willing to relocate candidates only considered.
2. One more information acutally needed as per the JD i.e. VISA sponsership required for the candidate or not.

However we don't have information about 2 in the dataset, hence used only 1st condition to create dataset

In [ ]:
#Creating data from json file

json_file = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/candidates.jsonl"

profiles = []
career_history = []
education = []
skills = []
certifications = []
languages = []
redrob_signals = []
industries = []
with open(json_file, "r", encoding="utf-8") as f:
    for line in f:
        candidate = json.loads(line)

        candidate_id = candidate["candidate_id"]

        profile = candidate.get("profile", {})
        signal = candidate.get("redrob_signals", {})
      #Filter the candidates who are not in India.
        if((profile.get("country") != "India")):
          continue
      #Filter the candidates who are not willing to relocate (There is VISA sponsership required information hence only used this).
        if (signal.get("willing_to_relocate") == False ):
          continue

        # -------------------------------
        # Candidate Profile
        # -------------------------------
        profiles.append({
            "candidate_id": candidate_id,
            "name": profile.get("anonymized_name"),
            "headline": profile.get("headline"),
            "summary": profile.get("summary"),
            "location": profile.get("location"),
            "country": profile.get("country"),
            "years_of_exp": profile.get("years_of_experience"),
            "current_title": profile.get("current_title"),
            "current_company": profile.get("current_company"),
            "current_company_size": profile.get("current_company_size"),
            "current_industry": profile.get("current_industry")
        })

        # Industry from current profile
        industries.append({
            "candidate_id": candidate_id,
            "Type_of_org": profile.get("current_industry"),
            "company_size": profile.get("current_company_size"),

        })
        candidate_id = candidate.get("candidate_id")
        companies = []
        titles = []
        start_dates = []
        end_dates = []
        industries = []
        company_sizes = []
        experience_descriptions = []

        # -------------------------------
        # Career History
        # -------------------------------
        for exp in candidate.get("career_history", []):
          # Convert months to years
          duration_months = exp.get("duration_months", 0)


          try:
               years = round(float(duration_months) / 12, 1)
          except:
               years = 0

          description = exp.get("description", "")

          enhanced_description = (
             f"{description} Worked here for about {years} years."
          ).strip()

          companies.append(str(exp.get("company", "")))
          titles.append(str(exp.get("title", "")))
          start_dates.append(str(exp.get("start_date", "")))
          end_dates.append(str(exp.get("end_date", "")))
          industries.append(str(exp.get("industry", "")))
          company_sizes.append(str(exp.get("company_size", "")))
          experience_descriptions.append(enhanced_description)

        career_history.append({
            "candidate_id": candidate_id,
            "companies": ", ".join(companies),
            "titles": ", ".join(titles),
            "start_dates": ", ".join(start_dates),
            "end_dates": ", ".join(end_dates),
            "industries": ", ".join(industries),
            "company_sizes": ", ".join(company_sizes),
             "career_history": " | ".join(experience_descriptions)
            })


            # Industry from career history
        industries.append({
                "candidate_id": candidate_id,
                "Type_of_org": exp.get("industry"),
                "company_size": exp.get("company_size"),
            })

        # -------------------------------
        # Education
        # -------------------------------
        for edu in candidate.get("education", []):
            education.append({
                "candidate_id": candidate_id,
                "institution": edu.get("institution"),
                "degree": edu.get("degree"),
                "field_of_study": edu.get("field_of_study"),
                "start_year": edu.get("start_year"),
                "end_year": edu.get("end_year"),
                "grade": edu.get("grade"),
                "tier": edu.get("tier")
            })

        # -------------------------------
        # Skills
        # -------------------------------
        candidate_skills = []
        for skill in candidate.get("skills", []):
          try:
              years = round(float(skill.get("duration_months", 0)) / 12, 1)
          except:
              years = 0

          skill_text = (
            f"{skill.get('name', '')} "
            f"({skill.get('proficiency', 'Unknown')}, "
            f"{skill.get('endorsements', 0)} endorsements, "
            f"{years} years)"
          )

          candidate_skills.append(skill_text)


        skills.append({
          "candidate_id": candidate_id,
          "skills": ", ".join(candidate_skills)
       })

        # -------------------------------
        # Certifications
        # -------------------------------
        for cert in candidate.get("certifications", []):
            certifications.append({
                "candidate_id": candidate_id,
                "name": cert.get("name"),
                "issuer": cert.get("issuer"),
                "year": cert.get("year")
            })

        # -------------------------------
        # Languages
        # -------------------------------
        for lang in candidate.get("languages", []):
            languages.append({
                "candidate_id": candidate_id,
                "language": lang.get("language"),
                "proficiency": lang.get("proficiency")
            })

        # -------------------------------
        # Redrob Signals
        # -------------------------------


        redrob_signals.append({
            "candidate_id": candidate_id,
            "profile_completeness_score": signal.get("profile_completeness_score"),
            "signup_date": signal.get("signup_date"),
            "last_active_date": signal.get("last_active_date"),
            "open_to_work_flag": signal.get("open_to_work_flag"),
            "profile_views_received_30d": signal.get("profile_views_received_30d"),
            "applications_submitted_30d": signal.get("applications_submitted_30d"),
            "recruiter_response_rate": signal.get("recruiter_response_rate"),
            "avg_response_time_hours": signal.get("avg_response_time_hours"),
            "skill_assessment_scores": json.dumps(
                signal.get("skill_assessment_scores", {})
            ),
            "connection_count": signal.get("connection_count"),
            "endorsements_received": signal.get("endorsements_received"),
            "notice_period_days": signal.get("notice_period_days"),
            "expected_min_salary_lpa":
                signal.get("expected_salary_range_inr_lpa", {}).get("min"),
            "expected_max_salary_lpa":
                signal.get("expected_salary_range_inr_lpa", {}).get("max"),
            "preferred_work_mode": signal.get("preferred_work_mode"),
            "willing_to_relocate": signal.get("willing_to_relocate"),
            "github_activity_score": signal.get("github_activity_score"),
            "search_appearance_30d": signal.get("search_appearance_30d"),
            "saved_by_recruiters_30d": signal.get("saved_by_recruiters_30d"),
            "interview_completion_rate": signal.get("interview_completion_rate"),
            "offer_acceptance_rate": signal.get("offer_acceptance_rate"),
            "verified_email": signal.get("verified_email"),
            "verified_phone": signal.get("verified_phone"),
            "linkedin_connected": signal.get("linkedin_connected")
        })






# ==========================================
# Create CSV files
# ==========================================
datasetPath = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/"
pd.DataFrame(profiles).to_csv(
    datasetPath+"candidate_profile.csv", index=False)

pd.DataFrame(career_history).to_csv(
    datasetPath+"candidate_career_history.csv", index=False)

pd.DataFrame(education).to_csv(
    datasetPath+"candidate_education.csv", index=False)

pd.DataFrame(skills).to_csv(
    datasetPath+"candidate_skills.csv", index=False)

pd.DataFrame(certifications).to_csv(
    datasetPath+"candidate_certifications.csv", index=False)

pd.DataFrame(languages).to_csv(
    datasetPath+"candidate_languages.csv", index=False)

pd.DataFrame(redrob_signals).to_csv(
    datasetPath+"candidate_redrob_signals.csv", index=False)

pd.DataFrame(industries).to_csv(
    datasetPath+"candidate_org.csv", index=False)
print("All CSV files generated successfully.")

All CSV files generated successfully.


Creating Vector database for the candidates based on the skills, summary and career history

In [ ]:
CANDIDATE_PROFILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_profile.csv'
CANDIDATE_SKILLS = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_skills.csv'
CANDIDATE_CAREER_HISTORY = '/content/drive/MyDrive/Hackthon/IndiaRuns/version2/dataset/candidate_career_history.csv'

In [3]:
!pip uninstall -y faiss faiss-cpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 48.2 MB/s eta 0:00:00


In [4]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sentence-transformers-5.5.1


In [5]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

In [6]:
FAISS_INDEX_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/database/candSummaryvector_db.faiss"

METADATA_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/database/candSummarymetadata.pkl"

MODEL_NAME = "all-MiniLM-L6-v2"

In [ ]:



# =====================================================
# LOAD FILES
# =====================================================

profile_df = pd.read_csv(CANDIDATE_PROFILE)

skills_df = pd.read_csv(CANDIDATE_SKILLS)

career_df = pd.read_csv(CANDIDATE_CAREER_HISTORY)

print("Profile Rows :", len(profile_df))
print("Skills Rows  :", len(skills_df))
print("Career Rows  :", len(career_df))

# =====================================================
# KEEP REQUIRED COLUMNS
# =====================================================

skills_df = skills_df[[
    "candidate_id",
    "skills"
]]

career_df = career_df[[
    "candidate_id",
    "companies",
    "titles",
    "industries",
    "career_history"
]]

# =====================================================
# MERGE DATA
# =====================================================

df = profile_df.merge(
    skills_df,
    on="candidate_id",
    how="left"
)

df = df.merge(
    career_df,
    on="candidate_id",
    how="left"
)

# =====================================================
# HANDLE NULLS
# =====================================================

columns_to_fill = [
    "summary",
    "skills",
    "companies",
    "titles",
    "industries",
    "career_history"
]

for col in columns_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna("")

# =====================================================
# CREATE COMBINED TEXT
# =====================================================

df["combined_text"] = (
    "Summary: " + df["summary"].astype(str) +
    "\nSkills: " + df["skills"].astype(str) +
    "\nCompanies: " + df["companies"].astype(str) +
    "\nJob Titles: " + df["titles"].astype(str) +
    "\nIndustries: " + df["industries"].astype(str) +
    "\nCareer History: " + df["career_history"].astype(str)
)

texts = df["combined_text"].tolist()

print("Documents:", len(texts))

# =====================================================
# LOAD MODEL
# =====================================================

print("Loading embedding model...")

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE EMBEDDINGS
# =====================================================

print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print("Embedding Shape:", embeddings.shape)

# =====================================================
# CREATE FAISS INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

# =====================================================
# SAVE INDEX
# =====================================================

faiss.write_index(
    index,
    FAISS_INDEX_FILE
)

print("FAISS index saved")

# =====================================================
# SAVE METADATA
# =====================================================

metadata = df.to_dict("records")

with open(METADATA_FILE, "wb") as f:
    pickle.dump(metadata, f)

print("Metadata saved")
print("Done!")

Profile Rows : 21673
Skills Rows  : 21673
Career Rows  : 21673
Documents: 21673
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/678 [00:00<?, ?it/s]

Embedding Shape: (21673, 384)
Vectors Stored: 21673
FAISS index saved
Metadata saved
Done!


Getting Sematic score of candidates skills, summary and previous work experience with JD

In [7]:
!pip install -q transformers accelerate torch python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.7 MB/s eta 0:00:00


In [8]:
JD_FILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/job_description.docx'

In [9]:
from docx import Document
# =====================================================
# READ JD DOCX
# =====================================================

def read_docx(file_path):
    doc = Document(file_path)

    text = "\n".join(
        para.text.strip()
        for para in doc.paragraphs
        if para.text.strip()
    )

    return text

jd_text = read_docx(JD_FILE)

print("JD Length:", len(jd_text))
print(jd_text[:500])

# =====================================================
# LOAD MODEL
# =====================================================

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE JD EMBEDDING
# =====================================================

jd_embedding = model.encode(
    [jd_text],
    normalize_embeddings=True
).astype(np.float32)

JD Length: 9564
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)
Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# =====================================================
# LOAD FAISS
# =====================================================

index = faiss.read_index(FAISS_INDEX_FILE)

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("Candidates:", len(metadata))

# =====================================================
# SEARCH ALL CANDIDATES
# =====================================================

k = index.ntotal

scores, indices = index.search(
    jd_embedding,
    k
)

Candidates: 21673


In [12]:
# =====================================================
# BUILD RESULT CSV
# =====================================================

results = []

for rank, idx in enumerate(indices[0]):

    similarity = float(scores[0][rank])

    # cosine similarity -> score 0-1
    semantic_score = (similarity + 1) / 2


    candidate = metadata[idx]

    results.append({
        "candidate_id": candidate.get("candidate_id"),
        "semantic_similarity": round(similarity, 4),
        "semantic_score": semantic_score
    })

# =====================================================
# SAVE OUTPUT
# =====================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "semantic_score",
    ascending=False
)

OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores.csv"

result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(result_df.head(20))

Saved: /content/drive/MyDrive/Hackthon/IndiaRuns/version2/candidatesrankfiles/candidate_semantic_scores.csv
    candidate_id  semantic_similarity  semantic_score
0   CAND_0076006               0.6782        0.839080
1   CAND_0026624               0.6754        0.837698
2   CAND_0043416               0.6749        0.837430
3   CAND_0023359               0.6739        0.836959
4   CAND_0016695               0.6732        0.836609
5   CAND_0097572               0.6718        0.835924
6   CAND_0020781               0.6704        0.835207
7   CAND_0081203               0.6696        0.834787
8   CAND_0046822               0.6695        0.834764
9   CAND_0063737               0.6694        0.834708
10  CAND_0012656               0.6693        0.834647
11  CAND_0040259               0.6692        0.834578
12  CAND_0061887               0.6691        0.834550
13  CAND_0071763               0.6687        0.834362
14  CAND_0013618               0.6687        0.834336
15  CAND_0000211            